# 01 — Advanced Retrieval Strategies

**Track:** Intermediate · **Stage:** Retrieval Engineering

Retrieval quality depends on matching the query type to the retrieval signal. Exact identifiers (like **AX-774-B**) are often lexical problems. Abstract concepts (like **"supplier for the Atlas database"**) are semantic problems. 

In this deep dive, you will build and compare:
1. **Dense Retrieval:** Semantic similarity using vector embeddings.
2. **Sparse Retrieval:** Keyword matching using BM25.
3. **Hybrid Retrieval:** Fusing sparse and dense scores using Reciprocal Rank Fusion (RRF).
4. **Query Transformation:** Using LLMs to rewrite and expand user queries before retrieval.

## Setup: LangChain Retrievers

We will use LangChain's specialized retrievers to build these pipelines.

In [ ]:
# !pip install langchain langchain-community langchain-huggingface chromadb rank_bm25

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.llms.fake import FakeListLLM

def print_results(results):
    for i, doc in enumerate(results):
        print(f"[{i+1}] Source: {doc.metadata['source']}")
        print(f"{doc.page_content}\n")

## 1. Mock Enterprise Data

We need a dataset that exposes the weaknesses of both dense and sparse retrieval. 
- `doc1` is a vendor contract (semantic). 
- `doc2` contains strict alphanumeric identifiers (lexical).

In [ ]:
corpus = [
    Document(
        page_content="Vendor Agreement: DataStax is the primary supplier for the Atlas vector database ecosystem, providing SLA guarantees for 99.99% uptime.",
        metadata={"source": "vendor_datastax.md"}
    ),
    Document(
        page_content="Hardware Spec AX-774-B: The new server rack requires 220V power and dual redundant cooling units. Do not mix with AX-774-A.",
        metadata={"source": "hardware_specs.md"}
    ),
    Document(
        page_content="Atlas Project Guidelines: All new code must be reviewed by two senior engineers before merging into the main branch.",
        metadata={"source": "atlas_guidelines.md"}
    )
]

## 2. Dense Retrieval (Semantic)

Dense retrieval converts text to dense vectors. It is excellent at understanding meaning (e.g. "supplier" = "vendor") but terrible at exact keyword matching (e.g. distinguishing "AX-774-B" from "AX-774-A").

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(corpus, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("--- Dense Retrieval: Semantic Question ---")
print_results(dense_retriever.invoke("Who supplies our vector DB?"))

print("--- Dense Retrieval: Lexical Question (May Struggle) ---")
print_results(dense_retriever.invoke("What power does AX-774-B need?"))

## 3. Sparse Retrieval (BM25 / Keyword)

Sparse retrieval (like TF-IDF or BM25) creates sparse vectors based on word frequencies. It is excellent at exact identifiers but terrible at synonyms.

In [ ]:
bm25_retriever = BM25Retriever.from_documents(corpus)
bm25_retriever.k = 2

print("--- Sparse Retrieval: Lexical Question ---")
print_results(bm25_retriever.invoke("What power does AX-774-B need?"))

print("--- Sparse Retrieval: Semantic Question (May Struggle) ---")
# Notice that BM25 struggles here because 'supplies' does not match 'supplier', 
# and 'vector DB' does not exactly match 'vector database'.
print_results(bm25_retriever.invoke("Who supplies our vector DB?"))

## 4. Hybrid Retrieval (Ensemble)

In production, you rarely choose one or the other. You use both and fuse the results. LangChain's `EnsembleRetriever` uses Reciprocal Rank Fusion (RRF) to combine the results of multiple retrievers.

In [ ]:
# We weight the dense retriever slightly higher, but give BM25 enough weight to save exact matches.
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever], 
    weights=[0.6, 0.4]
)

print("--- Hybrid Retrieval: Semantic Question ---")
print_results(hybrid_retriever.invoke("Who supplies our vector DB?"))

print("--- Hybrid Retrieval: Lexical Question ---")
print_results(hybrid_retriever.invoke("What power does AX-774-B need?"))

## 5. Query Transformation (Multi-Query)

Users often write terrible search queries. A `MultiQueryRetriever` uses an LLM to generate multiple variants of the user's question, retrieves documents for all variants, and deduplicates the results. This drastically increases recall.

*(We use a FakeListLLM here to simulate the LLM rewriting the query.)*

In [ ]:
# Simulate the LLM outputting 3 rewritten variants of the query
mock_rewrites = [
    "Who is the vendor for the Atlas vector database?",
    "Which company supplies the Atlas ecosystem?",
    "Atlas vector DB supplier information."
]
llm = FakeListLLM(responses=["\n".join(mock_rewrites)])

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever, 
    llm=llm
)

# Set up logging to see the generated queries in action
import logging
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

print("--- Multi-Query Retrieval ---")
results = multi_query_retriever.invoke("Who supplies our vector DB?")
print_results(results)

## Reflection

1. **Cost vs. Recall:** Multi-query transformation increases recall but adds an LLM call *before* retrieval. This adds latency and cost. Is this acceptable for your use case?
2. **Vector DB Support:** While `EnsembleRetriever` does fusion in-memory on the Python side, enterprise vector databases like Qdrant, Pinecone, or Elasticsearch support native hybrid search. Native hybrid search is significantly faster because it evaluates sparse and dense scores directly on the database nodes before returning results.